# convT-kernel-axis-swap — worked example 3: Convert a ConvT kernel to conv layout in NumPy and label the axes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-kernel-axis-swap`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The same axis-order asymmetry shows up outside PyTorch: a transposed-conv kernel is `(IC, OC, KH, KW)` and a conv kernel is `(OC, IC, KH, KW)`. In NumPy the conversion is `np.transpose(w, (1, 0, 2, 3))` — the channel axes (0 and 1) trade places while the spatial axes (2, 3) stay in order. Element `w_conv[o, i]` must equal `w_convT[i, o]`.

## Worked solution

**Goal.** In pure NumPy, turn a transposed-conv kernel `(IC, OC, KH, KW)` into a conv kernel `(OC, IC, KH, KW)` and prove the per-filter slices are preserved.

**Step 1 — choose the permutation.** `np.transpose` takes the new axis order as a tuple of *old* indices. We want old axis 1 (OC) first, then old axis 0 (IC), then the spatial axes unchanged: `(1, 0, 2, 3)`. Listing `2, 3` last guarantees the kernel grid is never reordered.

**Step 2 — understand what moves.** Only the first two entries of the permutation are non-identity, so this is exactly a channel swap. NumPy returns a view with new strides; the underlying buffer is shared, which is why no values change — only their addressing does.

**Step 3 — state the invariant.** After the swap, the filter that mapped in-channel `i` to out-channel `o` lives at `w_conv[o, i]` instead of `w_convT[i, o]`. Checking `w_conv[1, 2] == w_convT[2, 1]` confirms the correspondence for one (out, in) pair.

**Step 4 — verify shape and values.** Shape goes from `(4, 6, 3, 3)` to `(6, 4, 3, 3)`, and the spot-checked slice is identical, so the conversion is correct and lossless.

In [ ]:
def np_convT_to_conv(w_convT):
    # (IC, OC, KH, KW) -> (OC, IC, KH, KW); swap channel axes, keep spatial
    return np.transpose(w_convT, (1, 0, 2, 3))

np.random.seed(0)
w_convT = np.random.randn(4, 6, 3, 3)   # (IC=4, OC=6, KH=3, KW=3)
w_conv = np_convT_to_conv(w_convT)
print('convT shape (IC,OC,KH,KW):', w_convT.shape)
print('conv  shape (OC,IC,KH,KW):', w_conv.shape)
print('slice (o=1,i=2) preserved:', np.allclose(w_conv[1, 2], w_convT[2, 1]))